In [ ]:
# sklearn Baseline Comparison
print("\n scikit-learn Baseline (small sample only)")

from sklearn.linear_model    import LogisticRegression as SKLogReg
from sklearn.ensemble        import RandomForestClassifier as SKRandomForest
from sklearn.tree            import DecisionTreeClassifier as SKDecisionTree
from sklearn.metrics         import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

pdf_sk = (df_feat
    .sample(False, 0.003, seed=42)
    .select("features", "label", "crime_category")
    .toPandas()
)
X_sk = np.vstack(pdf_sk["features"].apply(lambda v: v.toArray()).values)
y_sk = pdf_sk["label"].values.astype(int)

X_tr_sk, X_te_sk, y_tr_sk, y_te_sk = train_test_split(
    X_sk, y_sk, test_size=0.2, random_state=42, stratify=y_sk
)
print("sklearn train size:", X_tr_sk.shape[0], "| test size:", X_te_sk.shape[0])

start_sk_lr = time.perf_counter()
sk_lr = SKLogReg(max_iter=200, random_state=42, multi_class="multinomial")
sk_lr.fit(X_tr_sk, y_tr_sk)
sk_lr_acc = accuracy_score(y_te_sk, sk_lr.predict(X_te_sk))
sk_lr_f1  = f1_score(y_te_sk, sk_lr.predict(X_te_sk), average="weighted", zero_division=0)
sk_lr_time = time.perf_counter() - start_sk_lr

start_sk_dt = time.perf_counter()
sk_dt = SKDecisionTree(max_depth=8, random_state=42)
sk_dt.fit(X_tr_sk, y_tr_sk)
sk_dt_acc = accuracy_score(y_te_sk, sk_dt.predict(X_te_sk))
sk_dt_f1  = f1_score(y_te_sk, sk_dt.predict(X_te_sk), average="weighted", zero_division=0)
sk_dt_time = time.perf_counter() - start_sk_dt

start_sk_rf = time.perf_counter()
sk_rf = SKRandomForest(n_estimators=50, max_depth=8, random_state=42)
sk_rf.fit(X_tr_sk, y_tr_sk)
sk_rf_acc = accuracy_score(y_te_sk, sk_rf.predict(X_te_sk))
sk_rf_f1  = f1_score(y_te_sk, sk_rf.predict(X_te_sk), average="weighted", zero_division=0)
sk_rf_time = time.perf_counter() - start_sk_rf

sklearn_comparison = pd.DataFrame([
    {"framework": "PySpark MLlib", "model": "LogisticRegression",
     "data_size": "~6M rows", "accuracy": round(lr_acc, 4),
     "f1_score": round(lr_f1, 4), "train_time_sec": round(lr_time, 2), "scalable": "Yes"},
    {"framework": "scikit-learn",  "model": "LogisticRegression",
     "data_size": str(X_tr_sk.shape[0]) + " rows", "accuracy": round(sk_lr_acc, 4),
     "f1_score": round(sk_lr_f1, 4), "train_time_sec": round(sk_lr_time, 2), "scalable": "No"},
    {"framework": "PySpark MLlib", "model": "DecisionTree",
     "data_size": "~6M rows", "accuracy": round(dt_acc, 4),
     "f1_score": round(dt_f1, 4), "train_time_sec": round(dt_time, 2), "scalable": "Yes"},
    {"framework": "scikit-learn",  "model": "DecisionTree",
     "data_size": str(X_tr_sk.shape[0]) + " rows", "accuracy": round(sk_dt_acc, 4),
     "f1_score": round(sk_dt_f1, 4), "train_time_sec": round(sk_dt_time, 2), "scalable": "No"},
    {"framework": "PySpark MLlib", "model": "RandomForest",
     "data_size": "~6M rows", "accuracy": round(rf_acc, 4),
     "f1_score": round(rf_f1, 4), "train_time_sec": round(rf_time, 2), "scalable": "Yes"},
    {"framework": "scikit-learn",  "model": "RandomForest",
     "data_size": str(X_tr_sk.shape[0]) + " rows", "accuracy": round(sk_rf_acc, 4),
     "f1_score": round(sk_rf_f1, 4), "train_time_sec": round(sk_rf_time, 2), "scalable": "No"},
])
print("\nPySpark vs scikit-learn:")
print(sklearn_comparison.to_string(index=False))
sklearn_comparison.to_csv(OUT_DIR + "/dash2_pyspark_vs_sklearn.csv", index=False)
print("Saved: dash2_pyspark_vs_sklearn.csv")


In [ ]:
#Bootstrap confidence intervals
def bootstrap_accuracy(pred_df, n_bootstrap=10, sample_frac=0.1, seed_base=0):
    scores = []
    for i in range(n_bootstrap):
        s     = pred_df.sample(False, sample_frac, seed=seed_base + i)
        score = float(acc_eval.evaluate(s))
        scores.append(score)
    arr = np.array(scores)
    return {
        "mean":     round(float(arr.mean()), 4),
        "ci_lower": round(float(np.percentile(arr, 2.5)), 4),
        "ci_upper": round(float(np.percentile(arr, 97.5)), 4),
        "std":      round(float(arr.std()), 4)
    }

print("\nBootstrap Confidence Intervals (accuracy) for multi-class models:")
lr_ci  = bootstrap_accuracy(lr_pred)
dt_ci  = bootstrap_accuracy(dt_pred)
rf_ci  = bootstrap_accuracy(rf_pred)
print("Logistic Regression:", lr_ci)
print("Decision Tree:      ", dt_ci)
print("Random Forest:      ", rf_ci)
print("GBT (binary AUC):", round(gbt_auc, 4), "(no bootstrap - single AUC metric)")

pd.DataFrame([
    {"model": "LogisticRegression", **lr_ci},
    {"model": "DecisionTree",       **dt_ci},
    {"model": "RandomForest",       **rf_ci},
    {"model": "GBT_Binary",
     "mean": round(gbt_auc, 4), "ci_lower": None, "ci_upper": None, "std": None}
]).to_csv(OUT_DIR + "/dash2_bootstrap_ci.csv", index=False)
print("Saved: dash2_bootstrap_ci.csv")

In [ ]:
#  Confusion matrix for best performing model
conf_matrix = (rf_pred
    .groupBy("label", "prediction")
    .count()
    .withColumnRenamed("label",      "actual_index")
    .withColumnRenamed("prediction", "predicted_index")
    .toPandas()
)
conf_matrix["actual_category"]    = conf_matrix["actual_index"].apply(
    lambda i: label_list[int(i)] if int(i) < len(label_list) else "UNKNOWN"
)
conf_matrix["predicted_category"] = conf_matrix["predicted_index"].apply(
    lambda i: label_list[int(i)] if int(i) < len(label_list) else "UNKNOWN"
)
conf_matrix.to_csv(OUT_DIR + "/dash2_rf_confusion_matrix.csv", index=False)
print("Saved: dash2_rf_confusion_matrix.csv")

# Overall model comparison table

model_comparison = pd.DataFrame([
    {"model": "Logistic Regression", "task": "3-class",
     "framework": "PySpark MLlib",
     "params": "regParam=0.001, maxIter=100",
     "primary_metric": "accuracy", "primary_value": round(lr_acc, 4),
     "f1_score": round(lr_f1, 4),
     "train_time_sec": round(lr_time, 2), "tuned": "No"},
    {"model": "Decision Tree", "task": "3-class",
     "framework": "PySpark MLlib",
     "params": "maxDepth=15, maxBins=128",
     "primary_metric": "accuracy", "primary_value": round(dt_acc, 4),
     "f1_score": round(dt_f1, 4),
     "train_time_sec": round(dt_time, 2), "tuned": "No"},
    {"model": "Random Forest", "task": "3-class",
     "framework": "PySpark MLlib",
     "params": "numTrees=150, maxDepth=15",
     "primary_metric": "accuracy", "primary_value": round(rf_acc, 4),
     "f1_score": round(rf_f1, 4),
     "train_time_sec": round(rf_time, 2), "tuned": "No"},
    {"model": "GBT (Binary)", "task": "VIOLENT vs Other",
     "framework": "PySpark MLlib",
     "params": "maxIter=30, maxDepth=8, stepSize=0.05",
     "primary_metric": "AUC-ROC", "primary_value": round(gbt_auc, 4),
     "f1_score": round(gbt_f1, 4),
     "train_time_sec": round(gbt_time, 2), "tuned": "No"},
    {"model": best_model_name + " (Tuned)", "task": "5-class (tuned)",
    {"model": "Best: " + best_model_name, "task": "3-class",
     "framework": "PySpark MLlib",
     "params": "Manually tuned (see report)",
     "primary_metric": "accuracy",
     "primary_value": round(initial_results[best_model_name], 4),
     "f1_score": round(initial_results[best_model_name], 4),
     "train_time_sec": "N/A", "tuned": "Manual"},
print("\nFull Model Comparison (including tuned):")
print(model_comparison.to_string(index=False))
model_comparison.to_csv(OUT_DIR + "/dash2_model_comparison.csv", index=False)
print("Saved: dash2_model_comparison.csv")

In [ ]:
# 7C - Confusion matrix for best performing model
conf_matrix = (rf_pred
    .groupBy("label", "prediction")
    .count()
    .withColumnRenamed("label",      "actual_index")
    .withColumnRenamed("prediction", "predicted_index")
    .toPandas()
)
conf_matrix["actual_category"]    = conf_matrix["actual_index"].apply(
    lambda i: label_list[int(i)] if int(i) < len(label_list) else "UNKNOWN"
)
conf_matrix["predicted_category"] = conf_matrix["predicted_index"].apply(
    lambda i: label_list[int(i)] if int(i) < len(label_list) else "UNKNOWN"
)
conf_matrix.to_csv(OUT_DIR + "/dash2_rf_confusion_matrix.csv", index=False)
print("Saved: dash2_rf_confusion_matrix.csv")



In [ ]:
# sklearn Baseline Comparison
print("\n scikit-learn Baseline (small sample only)")

from sklearn.linear_model    import LogisticRegression as SKLogReg
from sklearn.ensemble        import RandomForestClassifier as SKRandomForest
from sklearn.tree            import DecisionTreeClassifier as SKDecisionTree
from sklearn.metrics         import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

pdf_sk = (df_feat
    .sample(False, 0.003, seed=42)
    .select("features", "label", "crime_category")
    .toPandas()
)
X_sk = np.vstack(pdf_sk["features"].apply(lambda v: v.toArray()).values)
y_sk = pdf_sk["label"].values.astype(int)

X_tr_sk, X_te_sk, y_tr_sk, y_te_sk = train_test_split(
    X_sk, y_sk, test_size=0.2, random_state=42, stratify=y_sk
)
print("sklearn train size:", X_tr_sk.shape[0], "| test size:", X_te_sk.shape[0])

start_sk_lr = time.perf_counter()
sk_lr = SKLogReg(max_iter=200, random_state=42, multi_class="multinomial")
sk_lr.fit(X_tr_sk, y_tr_sk)
sk_lr_acc = accuracy_score(y_te_sk, sk_lr.predict(X_te_sk))
sk_lr_f1  = f1_score(y_te_sk, sk_lr.predict(X_te_sk), average="weighted", zero_division=0)
sk_lr_time = time.perf_counter() - start_sk_lr

start_sk_dt = time.perf_counter()
sk_dt = SKDecisionTree(max_depth=8, random_state=42)
sk_dt.fit(X_tr_sk, y_tr_sk)
sk_dt_acc = accuracy_score(y_te_sk, sk_dt.predict(X_te_sk))
sk_dt_f1  = f1_score(y_te_sk, sk_dt.predict(X_te_sk), average="weighted", zero_division=0)
sk_dt_time = time.perf_counter() - start_sk_dt

start_sk_rf = time.perf_counter()
sk_rf = SKRandomForest(n_estimators=50, max_depth=8, random_state=42)
sk_rf.fit(X_tr_sk, y_tr_sk)
sk_rf_acc = accuracy_score(y_te_sk, sk_rf.predict(X_te_sk))
sk_rf_f1  = f1_score(y_te_sk, sk_rf.predict(X_te_sk), average="weighted", zero_division=0)
sk_rf_time = time.perf_counter() - start_sk_rf

sklearn_comparison = pd.DataFrame([
    {"framework": "PySpark MLlib", "model": "LogisticRegression",
     "data_size": "~6M rows", "accuracy": round(lr_acc, 4),
     "f1_score": round(lr_f1, 4), "train_time_sec": round(lr_time, 2), "scalable": "Yes"},
    {"framework": "scikit-learn",  "model": "LogisticRegression",
     "data_size": str(X_tr_sk.shape[0]) + " rows", "accuracy": round(sk_lr_acc, 4),
     "f1_score": round(sk_lr_f1, 4), "train_time_sec": round(sk_lr_time, 2), "scalable": "No"},
    {"framework": "PySpark MLlib", "model": "DecisionTree",
     "data_size": "~6M rows", "accuracy": round(dt_acc, 4),
     "f1_score": round(dt_f1, 4), "train_time_sec": round(dt_time, 2), "scalable": "Yes"},
    {"framework": "scikit-learn",  "model": "DecisionTree",
     "data_size": str(X_tr_sk.shape[0]) + " rows", "accuracy": round(sk_dt_acc, 4),
     "f1_score": round(sk_dt_f1, 4), "train_time_sec": round(sk_dt_time, 2), "scalable": "No"},
    {"framework": "PySpark MLlib", "model": "RandomForest",
     "data_size": "~6M rows", "accuracy": round(rf_acc, 4),
     "f1_score": round(rf_f1, 4), "train_time_sec": round(rf_time, 2), "scalable": "Yes"},
    {"framework": "scikit-learn",  "model": "RandomForest",
     "data_size": str(X_tr_sk.shape[0]) + " rows", "accuracy": round(sk_rf_acc, 4),
     "f1_score": round(sk_rf_f1, 4), "train_time_sec": round(sk_rf_time, 2), "scalable": "No"},
])
print("\nPySpark vs scikit-learn:")
print(sklearn_comparison.to_string(index=False))
sklearn_comparison.to_csv(OUT_DIR + "/dash2_pyspark_vs_sklearn.csv", index=False)
print("Saved: dash2_pyspark_vs_sklearn.csv")


In [ ]:
# Strong Scaling (fixed data size, vary partitions/parallelism)
print("Strong Scaling")
fixed_df   = df_feat.sample(False, 0.25, seed=42).cache()
fixed_df.count()
fixed_size = fixed_df.count()
print("Fixed dataset size:", fixed_size, "rows")

strong_results = []
for num_partitions in [50, 100, 200]:
    rdf   = fixed_df.repartition(num_partitions)
    start = time.perf_counter()
    m     = RandomForestClassifier(
                featuresCol="features", labelCol="label",
                numTrees=50, maxDepth=5, seed=42
            ).fit(rdf)
    pred  = m.transform(rdf)
    acc   = float(acc_eval.evaluate(pred))
    elapsed = time.perf_counter() - start
    strong_results.append({
        "scaling_type": "strong",
        "partitions":   num_partitions,
        "rows":         fixed_size,
        "time_seconds": round(elapsed, 2),
        "accuracy":     round(acc, 4)
    })
    print("Partitions:", num_partitions,
          "| Time:", round(elapsed, 2), "s",
          "| Accuracy:", round(acc, 4))

# 8B - Weak Scaling (data grows with partitions)
print("\n Weak Scaling ")
weak_results = []
for frac, num_partitions in [(0.25, 50), (0.50, 100), (0.75, 200)]:
    wdf   = df_feat.sample(False, frac, seed=42).repartition(num_partitions)
    n     = wdf.count()
    wdf   = wdf.cache()
    wdf.count()
    start = time.perf_counter()
    m     = RandomForestClassifier(
                featuresCol="features", labelCol="label",
                numTrees=50, maxDepth=5, seed=42
            ).fit(wdf)
    elapsed = time.perf_counter() - start
    weak_results.append({
        "scaling_type": "weak",
        "partitions":   num_partitions,
        "rows":         n,
        "time_seconds": round(elapsed, 2),
        "accuracy":     None
    })
    print("Fraction:", frac,
          "| Rows:", n,
          "| Partitions:", num_partitions,
          "| Time:", round(elapsed, 2), "s")
    wdf.unpersist()

# 8C - Caching Strategy Comparison
print("\n Cache vs No Cache ")
test_s = df_feat.sample(False, 0.15, seed=42)

t1 = time.perf_counter()
RandomForestClassifier(
    featuresCol="features", labelCol="label",
    numTrees=30, maxDepth=5, seed=42
).fit(test_s)
time_no_cache = time.perf_counter() - t1

cached_s = test_s.cache()
cached_s.count()
t2 = time.perf_counter()
RandomForestClassifier(
    featuresCol="features", labelCol="label",
    numTrees=30, maxDepth=5, seed=42
).fit(cached_s)
time_with_cache = time.perf_counter() - t2
cached_s.unpersist()

print("Without cache:", round(time_no_cache, 2), "s")
print("With cache:   ", round(time_with_cache, 2), "s")

# 8D - Partition distribution analysis
part_sizes  = (df_feat
    .withColumn("part_id", F.spark_partition_id())
    .groupBy("part_id")
    .count()
)
part_stats_row = part_sizes.agg(
    F.min("count").alias("min_rows"),
    F.max("count").alias("max_rows"),
    F.avg("count").alias("avg_rows"),
    F.stddev("count").alias("stddev_rows"),
    F.count("*").alias("total_partitions")
).collect()[0]

partition_stats = pd.DataFrame([{
    "min_rows_per_partition":    int(part_stats_row["min_rows"]),
    "max_rows_per_partition":    int(part_stats_row["max_rows"]),
    "avg_rows_per_partition":    round(float(part_stats_row["avg_rows"]), 0),
    "stddev_rows_per_partition": round(float(part_stats_row["stddev_rows"]), 2),
    "total_partitions":          int(part_stats_row["total_partitions"]),
    "skew_ratio":                round(
        float(part_stats_row["max_rows"]) / float(part_stats_row["avg_rows"]), 2
    )
}])
print("\nPartition stats:")
print(partition_stats.to_string(index=False))

# Save scalability exports
all_scaling = pd.DataFrame(strong_results + weak_results)
all_scaling["rows_per_second"] = (
    all_scaling["rows"] / all_scaling["time_seconds"]
).round(0)
all_scaling.to_csv(OUT_DIR + "/dash4_scalability_results.csv", index=False)
print("Saved: dash4_scalability_results.csv")

pd.DataFrame([
    {"strategy": "no_cache",   "time_seconds": round(time_no_cache, 2)},
    {"strategy": "with_cache", "time_seconds": round(time_with_cache, 2)}
]).to_csv(OUT_DIR + "/dash4_cache_comparison.csv", index=False)
print("Saved: dash4_cache_comparison.csv")

partition_stats.to_csv(OUT_DIR + "/dash4_partition_stats.csv", index=False)
print("Saved: dash4_partition_stats.csv")

pd.DataFrame([{
    "config_key": "executor_memory",     "value": "4g",
    "justification": "Prevents OOM for 6M row feature vectors"
},{
    "config_key": "executor_cores",      "value": "2",
    "justification": "Balances parallelism with Colab resource limits"
},{
    "config_key": "shuffle_partitions",  "value": "200",
    "justification": "Optimal for ~6M row dataset"
},{
    "config_key": "adaptive_execution",  "value": "enabled",
    "justification": "Auto-coalesces small partitions and picks join strategy"
},{
    "config_key": "storage_format",      "value": "Parquet",
    "justification": "Columnar, compressed, supports predicate pushdown"
},{
    "config_key": "partition_column",    "value": "Year",
    "justification": "Time-range queries skip irrelevant year partitions"
}]).to_csv(OUT_DIR + "/dash4_resource_config.csv", index=False)
print("Saved: dash4_resource_config.csv")

pd.DataFrame([{
    "bottleneck": "I/O",
    "description": "Initial parquet conversion from HuggingFace format",
    "mitigation":  "Partitioned Parquet with Year enables predicate pushdown"
},{
    "bottleneck": "Computation",
    "description": "Multi-class CrossValidator: 2 folds x 2 params = 4 fits",
    "mitigation":  "parallelism=4 runs folds concurrently; reduced grid size"
},{
    "bottleneck": "Memory",
    "description": "8M row feature vector materialisation across 4 models",
    "mitigation":  "persist() on df_feat avoids recomputation"
},{
    "bottleneck": "Shuffle",
    "description": "GroupBy and join operations on 8M rows",
    "mitigation":  "Broadcast join for lookup table; adaptive shuffle sizing"
}]).to_csv(OUT_DIR + "/dash4_bottleneck_summary.csv", index=False)
print("Saved: dash4_bottleneck_summary.csv")

pd.DataFrame([
    {"model": "LogisticRegression", "train_time_sec": round(lr_time, 2),
     "accuracy": round(lr_acc, 4), "f1_score": round(lr_f1, 4)},
    {"model": "DecisionTree",       "train_time_sec": round(dt_time, 2),
     "accuracy": round(dt_acc, 4), "f1_score": round(dt_f1, 4)},
    {"model": "RandomForest",       "train_time_sec": round(rf_time, 2),
     "accuracy": round(rf_acc, 4), "f1_score": round(rf_f1, 4)},
    {"model": "GBT_Binary",          "train_time_sec": round(gbt_time, 2),
     "accuracy": round(gbt_auc, 4), "f1_score": round(gbt_f1, 4)},  # AUC-ROC used as accuracy equivalent
]).to_csv(OUT_DIR + "/dash4_training_times.csv", index=False)